In [ ]:
"""
MAP Competition - Qwen3-4B Inference with LoRA Adapter
Single model inference for Kaggle submission with Explainable AI
"""

import os
import gc
import json
import re
import warnings
from collections import Counter
from pathlib import Path

warnings.filterwarnings('ignore')

def log(msg):
    print(msg, flush=True)

log("Starting imports...")

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoModelForCausalLM, AutoTokenizer, DataCollatorWithPadding
from peft import PeftModel, PeftConfig
from scipy.special import softmax
from tqdm import tqdm

# ============================================================================
# CONFIGURATION
# ============================================================================

TRAIN_PATH = "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/train.csv"
TEST_PATH = "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/test.csv"

# Qwen3-4B Model Paths - will be resolved by find functions
# Kaggle converts dataset names to lowercase with hyphens
BASE_MODEL_CANDIDATES = [
    "/kaggle/input/qwen3-4b-base",
    "/kaggle/input/qwen3-4b_base",
    "/kaggle/input/qwen3-4b-base/qwen3-4b",
    "/kaggle/input/qwen3-4b_base/qwen3-4b",
    "/kaggle/input/qwen34b-base",
    "/kaggle/input/qwen3-4b",
]

ADAPTER_CANDIDATES = [
    "/kaggle/input/map-qwen3-4b-trained",
    "/kaggle/input/mapqwen3-4b-trained",
    "/kaggle/input/map-qwen34b-trained",
    # Direct paths to adapter location (if dataset structure is preserved)
    "/kaggle/input/map-qwen3-4b-trained/map_multi_llm_runs/Qwen3-4B-Instruct-2507/full_train/Qwen3-4B-Instruct-2507",
    "/kaggle/input/mapqwen3-4b-trained/map_multi_llm_runs/Qwen3-4B-Instruct-2507/full_train/Qwen3-4B-Instruct-2507",
]

MAX_LEN = 256
BATCH_SIZE = 4

# Explainable AI Configuration - uses the SAME Qwen3-4B base model
USE_LLM_EXPLAINER = True
USE_LLM_VERIFIER = False
KB_BUILD_TOP_N = 50
KB_EXAMPLES_PER_MISCON = 3
KB_PATH = Path("misconception_kb.json")

# Device configuration (defined at module level for all cells)
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def find_model_path(base_paths):
    """Find the actual model directory from a list of candidate paths."""
    import glob
    
    # Handle single path or list of paths
    if isinstance(base_paths, str):
        base_paths = [base_paths]
    
    for base_path in base_paths:
        if not os.path.exists(base_path):
            continue
            
        # Check base path first
        if os.path.exists(os.path.join(base_path, "config.json")):
            # Verify it has model weights
            if (os.path.exists(os.path.join(base_path, "model.safetensors")) or
                os.path.exists(os.path.join(base_path, "pytorch_model.bin")) or
                glob.glob(os.path.join(base_path, "model*.safetensors"))):
                return base_path

        # Search for config.json with model weights in subdirectories
        for depth in range(1, 5):
            pattern = os.path.join(base_path, "/".join(["*"] * depth), "config.json")
            matches = glob.glob(pattern)
            for m in matches:
                dir_path = os.path.dirname(m)
                has_weights = (os.path.exists(os.path.join(dir_path, "model.safetensors")) or
                              os.path.exists(os.path.join(dir_path, "pytorch_model.bin")) or
                              glob.glob(os.path.join(dir_path, "model*.safetensors")))
                if has_weights:
                    return dir_path
    
    # Return first candidate as fallback
    return base_paths[0] if base_paths else None


def find_adapter_path(base_paths):
    """Find the actual adapter directory from a list of candidate paths."""
    import glob
    
    # Handle single path or list of paths
    if isinstance(base_paths, str):
        base_paths = [base_paths]
    
    for base_path in base_paths:
        if not os.path.exists(base_path):
            continue
            
        # Check base path first
        if os.path.exists(os.path.join(base_path, "adapter_config.json")):
            return base_path

        # Search for adapter_config.json in subdirectories (up to 6 levels deep)
        for depth in range(1, 7):
            pattern = os.path.join(base_path, "/".join(["*"] * depth), "adapter_config.json")
            matches = glob.glob(pattern)
            for m in matches:
                return os.path.dirname(m)
    
    # Return first candidate as fallback
    return base_paths[0] if base_paths else None


def list_kaggle_inputs():
    """List all available Kaggle input directories for debugging."""
    input_dir = "/kaggle/input"
    if os.path.exists(input_dir):
        dirs = os.listdir(input_dir)
        log(f"Available Kaggle inputs: {dirs}")
        # Also list subdirectories for each input
        for d in dirs:
            subpath = os.path.join(input_dir, d)
            if os.path.isdir(subpath):
                try:
                    subdirs = os.listdir(subpath)[:5]
                    log(f"  {d}/: {subdirs}")
                except:
                    pass
        return dirs
    return []


def format_input(row):
    """Format input text for Qwen3-4B model."""
    x = "This answer is correct." if row['is_correct'] else "This answer is incorrect."
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"{x}\n"
        f"Student Explanation: {row['StudentExplanation']}"
    )


# ============================================================================
# MAIN INFERENCE
# ============================================================================

log("="*60)
log("MAP COMPETITION - QWEN3-4B INFERENCE (with LoRA)")
log("="*60)

if torch.cuda.is_available():
    log(f"GPU: {torch.cuda.get_device_name(0)}")

# List available inputs for debugging
available_inputs = list_kaggle_inputs()

# Load data
log("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

log(f"Train: {train_df.shape}, Test: {test.shape}")

# Prepare labels
train_df['Misconception'] = train_df['Misconception'].fillna('NA')
train_df['target'] = train_df['Category'] + ':' + train_df['Misconception']
train_df['label_str'] = train_df['target']  # Alias for explainability

le = LabelEncoder()
le.fit(train_df['target'])
n_classes = len(le.classes_)
log(f"Number of classes: {n_classes}")

# Find correct answers for is_correct flag
idx = train_df['Category'].apply(lambda c: c.split('_')[0]) == 'True'
correct = train_df.loc[idx].copy()
correct['c'] = correct.groupby(['QuestionId', 'MC_Answer'])['MC_Answer'].transform('count')
correct = correct.sort_values('c', ascending=False).drop_duplicates(['QuestionId'])
correct = correct[['QuestionId', 'MC_Answer']]
correct['is_correct'] = 1

# Build correct_by_q_full dictionary for explainability
correct_by_q_full = dict(zip(correct['QuestionId'], correct['MC_Answer']))

# Merge is_correct into test
test = test.merge(correct, on=['QuestionId', 'MC_Answer'], how='left')
test['is_correct'] = test['is_correct'].fillna(0)

# Merge is_correct into train_df for explainability
train_df = train_df.merge(correct[['QuestionId', 'MC_Answer', 'is_correct']], 
                           on=['QuestionId', 'MC_Answer'], how='left')
train_df['is_correct'] = train_df['is_correct'].fillna(0)

# Format input text
test['text'] = test.apply(format_input, axis=1)

# Build family map for filtering
fam_map = {}
for _, row in test.iterrows():
    fam_map[row['row_id']] = 'True_' if row['is_correct'] == 1 else 'False_'

# Find model paths - search through candidates
log("Searching for base model...")
base_model_path = find_model_path(BASE_MODEL_CANDIDATES)
log(f"Found base model at: {base_model_path}")

log("Searching for adapter...")
adapter_path = find_adapter_path(ADAPTER_CANDIDATES)
log(f"Found adapter at: {adapter_path}")

# Verify paths exist
if not base_model_path or not os.path.exists(base_model_path):
    log("ERROR: Could not find base model!")
    log("Please check your Kaggle dataset names above and update BASE_MODEL_CANDIDATES")
    raise FileNotFoundError(f"Base model path not found. Searched: {BASE_MODEL_CANDIDATES}")

if not adapter_path or not os.path.exists(adapter_path):
    log("ERROR: Could not find adapter!")
    log("Please check your Kaggle dataset names above and update ADAPTER_CANDIDATES")
    raise FileNotFoundError(f"Adapter path not found. Searched: {ADAPTER_CANDIDATES}")

# Store base model path globally for explainer
BASE_MODEL_PATH_RESOLVED = base_model_path

# Load tokenizer from base model (use local_files_only=True for local paths)
log("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    base_model_path, 
    trust_remote_code=True,
    local_files_only=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base model for classification
log("Loading Qwen3-4B base model...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_path,
    num_labels=n_classes,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map=device,
    local_files_only=True
)
base_model.config.pad_token_id = tokenizer.pad_token_id

# Load LoRA adapter
log("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path, local_files_only=True)
model.eval()
log("Model with LoRA adapter loaded!")

# Tokenize
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

ds_test = Dataset.from_pandas(test[['text']])
ds_test = ds_test.map(tokenize, batched=True, remove_columns=['text'])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
dataloader = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

# Run inference
log(f"Running inference on {len(ds_test)} samples...")
all_logits = []
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Inference"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        all_logits.append(outputs.logits.float().cpu().numpy())

predictions = np.concatenate(all_logits, axis=0)
probs = softmax(predictions, axis=1)
top_indices = np.argsort(-probs, axis=1)

# Decode labels
flat_indices = top_indices.flatten()
decoded_labels = le.inverse_transform(flat_indices)
top_labels = decoded_labels.reshape(top_indices.shape)

# Apply family filter and create submission
log("Creating submission with family filtering...")
final_predictions = []

for i, row_id in enumerate(test.row_id.values):
    pref = fam_map.get(row_id, 'False_')

    # Filter predictions by family prefix
    filtered = [lbl for lbl in top_labels[i] if lbl.startswith(pref)]

    # Take top 3
    top3 = filtered[:3]

    # Fill if needed
    while len(top3) < 3:
        filler = f"{pref}Neither:NA"
        if filler not in top3:
            top3.append(filler)
        else:
            top3.append(f"{pref}Correct:NA" if pref == "True_" else f"{pref}Misconception:NA")

    final_predictions.append(" ".join(top3[:3]))

# Save submission
submission = pd.DataFrame({
    'row_id': test.row_id.values,
    'Category:Misconception': final_predictions
})
submission.to_csv('submission.csv', index=False)

log("\n" + "="*60)
log("SUBMISSION SAVED: submission.csv")
log("="*60)
log(str(submission.head(10)))
log(f"\nTotal predictions: {len(submission)}")

# Clean up classification model to free memory for CausalLM explainer
log("\nCleaning up classification model for explainer...")
del model
del base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Cell 8 — Explainable AI: Misconception Knowledge Base

A key differentiator of this pipeline is **explainability**. We build a **Misconception Knowledge Base (KB)** using the **same Qwen3-4B base model**:

1. For each misconception type (e.g., "Additive", "Duplication"), sample representative student explanations.
2. Load the Qwen3-4B base model from `BASE_MODEL_PATH_RESOLVED` as a CausalLM (for text generation).
3. Prompt it to generate a structured "knowledge card":
   - Plain definition
   - Typical wrong thought patterns
   - Correct rule
   - Fix steps for teachers
   - Quick self-check question
4. Cache the KB as JSON for reuse.

This enables the pipeline to not only **predict** misconceptions but also **explain** them to teachers — using a single model family throughout.

In [ ]:
# ============================================================
# EXPLAINABLE AI — MISCONCEPTION KNOWLEDGE BASE
# Uses the SAME Qwen3-4B base model (loaded as CausalLM)
# ============================================================

def load_explainer():
    """
    Load the SAME Qwen3-4B base model as CausalLM for text generation.
    Tries float16 on GPU first, then falls back to other options.
    """
    if not USE_LLM_EXPLAINER:
        return None, None

    print(f"Loading Qwen3-4B explainer from: {BASE_MODEL_PATH_RESOLVED}")
    
    # Verify path exists
    if not os.path.exists(BASE_MODEL_PATH_RESOLVED):
        print(f"ERROR: Path does not exist: {BASE_MODEL_PATH_RESOLVED}")
        return None, None
    
    # Load tokenizer from the base model
    tok = AutoTokenizer.from_pretrained(
        BASE_MODEL_PATH_RESOLVED, 
        trust_remote_code=True,
        local_files_only=True
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    
    mdl = None
    
    # Try 4-bit quantization first (if bitsandbytes is available)
    if "cuda" in device:
        try:
            from transformers import BitsAndBytesConfig
            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )
            mdl = AutoModelForCausalLM.from_pretrained(
                BASE_MODEL_PATH_RESOLVED,
                quantization_config=bnb_cfg,
                device_map="auto",
                torch_dtype=torch.float16,
                trust_remote_code=True,
                local_files_only=True,
            )
            mdl.eval()
            print(f"Successfully loaded Qwen3-4B explainer (4-bit quantized)")
            return mdl, tok
        except Exception as e:
            print(f"4-bit quantization failed: {str(e)[:100]}")
            print("Trying float16 without quantization...")
    
    # Fallback: Load without quantization (float16 on GPU)
    if "cuda" in device:
        try:
            mdl = AutoModelForCausalLM.from_pretrained(
                BASE_MODEL_PATH_RESOLVED,
                device_map="auto",
                torch_dtype=torch.float16,
                trust_remote_code=True,
                local_files_only=True,
            )
            mdl.eval()
            print(f"Successfully loaded Qwen3-4B explainer (float16, no quantization)")
            return mdl, tok
        except Exception as e:
            print(f"Float16 loading failed: {str(e)[:100]}")
    
    # CPU fallback (will be slower)
    try:
        mdl = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_PATH_RESOLVED,
            torch_dtype=torch.float32,
            trust_remote_code=True,
            low_cpu_mem_usage=True,
            local_files_only=True,
        )
        mdl.eval()
        print(f"Successfully loaded Qwen3-4B explainer (CPU, float32)")
        return mdl, tok
    except Exception as e:
        print(f"ERROR: All loading methods failed: {e}")
        print("Explainable AI will use rule-based fallback only.")
        return None, None


def llm_generate(prompt: str, max_new_tokens=320, explainer_mdl=None, explainer_tok=None):
    """Generate text using the Qwen3-4B model."""
    if explainer_mdl is None:
        return ""
    
    # Format prompt for Qwen3 (uses chat template)
    messages = [{"role": "user", "content": prompt}]
    try:
        chat_prompt = explainer_tok.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True,
            enable_thinking=False  # Disable thinking for faster response
        )
    except TypeError:
        # Fallback if enable_thinking is not supported
        chat_prompt = explainer_tok.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
    
    inputs = explainer_tok(chat_prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(explainer_mdl.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        out = explainer_mdl.generate(
            **inputs, 
            max_new_tokens=max_new_tokens, 
            do_sample=False, 
            num_beams=1,
            pad_token_id=explainer_tok.eos_token_id
        )
    
    # Decode and extract only the generated part
    full_text = explainer_tok.decode(out[0], skip_special_tokens=True)
    # Remove the input prompt from the output
    input_text = explainer_tok.decode(inputs['input_ids'][0], skip_special_tokens=True)
    if full_text.startswith(input_text):
        text = full_text[len(input_text):].strip()
    else:
        text = full_text.strip()
    
    return text


# Load the Qwen3-4B base model for explainable AI
explainer_model, explainer_tok = load_explainer()

# ============================================================
# BUILD MISCONCEPTION KNOWLEDGE BASE
# ============================================================

train_df["miscon_only"] = train_df["Misconception"].astype(str)
freq = Counter(train_df["miscon_only"].tolist())
top_miscons = [m for m, _ in freq.most_common(KB_BUILD_TOP_N) if m != "NA"]
print(f"\nBuilding KB for {len(top_miscons)} misconception types...")

if KB_PATH.exists():
    kb = json.loads(KB_PATH.read_text(encoding="utf-8"))
    print(f"Loaded cached KB: {len(kb)} entries")
else:
    kb = {}
    for i, miscon in enumerate(top_miscons, start=1):
        sub = train_df[train_df["miscon_only"] == miscon]
        sample = sub.sample(n=min(KB_EXAMPLES_PER_MISCON, len(sub)), random_state=42)

        examples = []
        for _, r in sample.iterrows():
            examples.append({
                "question": str(r["QuestionText"])[:200],
                "student_answer": str(r["MC_Answer"]),
                "student_expl": str(r["StudentExplanation"])[:200],
            })

        prompt = f"""You are a math teacher writing a misconception knowledge card.

Misconception name: {miscon}

Student examples:
{json.dumps(examples, ensure_ascii=False, indent=2)}

Return ONLY JSON with keys:
plain_definition (1-2 sentences),
typical_wrong_thought (list of strings),
correct_rule (list of strings),
fix_steps (list of 4-7 steps),
quick_check (one question string),
tiny_example (2-4 line string).
"""
        out = llm_generate(prompt, 420, explainer_model, explainer_tok)

        card = None
        if out:
            match = re.search(r"\{.*\}", out, flags=re.S)
            if match:
                try:
                    card = json.loads(match.group(0))
                except Exception:
                    card = None

        if card is None:
            # Rule-based fallback when LLM generation fails
            card = {
                "plain_definition": f"Students make this mistake: {miscon}.",
                "typical_wrong_thought": ["They apply an incorrect rule or shortcut."],
                "correct_rule": ["Use the correct definition and check carefully."],
                "fix_steps": ["Restate the goal.", "Write the correct rule.", "Apply step-by-step.", "Check with an easy example."],
                "quick_check": "Can you test your rule on a simpler number?",
                "tiny_example": "Example: Compare decimals by place value, not by digit count."
            }

        kb[miscon] = card
        if i % 20 == 0:
            print(f"  Built {i}/{len(top_miscons)} cards...")

    KB_PATH.write_text(json.dumps(kb, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved KB: {len(kb)} entries to {KB_PATH}")

# Show a sample card
if kb:
    sample_key = next(iter(kb.keys()))
    print(f"\n--- Sample KB Card: '{sample_key}' ---")
    print(json.dumps(kb[sample_key], ensure_ascii=False, indent=2)[:800])
else:
    print("No KB entries built.")

## Cell 9 — Explainable AI: Teacher Feedback Generator

The final explainability component generates **personalized teacher feedback** for each student using the **same Qwen3-4B base model**:
- Uses the Qwen3-4B prediction (with LoRA) + KB card to produce targeted feedback
- Includes: what the student seems to think, the misconception in simple words, step-by-step correct logic, and a self-check question
- Rule-based verification ensures consistency (e.g., don't say "wrong" if the student was correct)
- Optional Qwen3-4B-based verification to fix contradictions

**Single Model Family Pipeline**: Classification uses Qwen3-4B + LoRA adapter, explanation generation uses Qwen3-4B base model.

In [ ]:
# ============================================================
# EXPLAINABLE AI — TEACHER FEEDBACK GENERATOR
# Uses the SAME Qwen3-4B base model loaded in Cell 8
# ============================================================

def get_correct_choice(qid):
    """Look up the inferred correct answer for a question."""
    return str(correct_by_q_full.get(qid, ""))


def rule_based_verify(feedback: str, is_correct: int, correct_choice: str) -> str:
    """
    Rule-based post-hoc check:
    - If student is correct, soften negative language
    - If student is incorrect and we know the answer, prepend it
    """
    txt = feedback.strip()
    if is_correct == 1:
        txt = re.sub(r"\bincorrect\b", "not quite right", txt, flags=re.I)
        txt = re.sub(r"\bwrong\b", "not quite right", txt, flags=re.I)
    if is_correct == 0 and correct_choice and correct_choice not in txt:
        txt = f"Correct answer: {correct_choice}\n\n" + txt
    return txt


def llm_verify(feedback: str) -> str:
    """Optional LLM-based verification using Qwen3-4B."""
    if not USE_LLM_VERIFIER or explainer_model is None:
        return feedback
    prompt = f"Fix contradictions and make this concise (8-14 lines), student-friendly:\n\n{feedback}"
    out = llm_generate(prompt, 260, explainer_model, explainer_tok)
    return out if out else feedback


def generate_teacher_feedback(row, pred_label: str, pred_top3: list) -> str:
    """
    Generate an explanation for the predicted misconception.
    Uses: Qwen3-4B prediction + Knowledge Base card + Qwen3-4B generation.
    """
    qtext   = str(row["QuestionText"])
    ans     = str(row["MC_Answer"])
    expl    = str(row["StudentExplanation"])
    qid     = row["QuestionId"]
    correct = get_correct_choice(qid)
    ic      = int(row.get("is_correct", -1))
    ic_str  = {1: "Yes", 0: "No"}.get(ic, "Unknown")

    miscon = pred_label.split(":", 1)[1] if ":" in pred_label else pred_label
    card   = kb.get(miscon, {})

    if USE_LLM_EXPLAINER and explainer_model is not None:
        prompt = f"""You are a kind, clear math teacher. Use ONLY the info below.

Question: {qtext[:300]}
Student chose: {ans}
Is correct? {ic_str}
Correct choice: {correct}
Student explanation: {expl[:300]}
Top predictions: {pred_top3}
Misconception knowledge card:
{json.dumps(card, ensure_ascii=False, indent=2)[:600]}

Write:
- 1–2 lines: what the student seems to think
- 1 line: misconception in simple words
- 4–7 short steps: correct logic
- 1 line: self-check question
Keep it concise and encouraging.
"""
        fb = llm_generate(prompt, 320, explainer_model, explainer_tok) or "Feedback generation failed."
    else:
        # Rule-based fallback when Qwen3-4B is not available
        fb = f"""Teacher Feedback
Student chose: {ans} | Correct: {ic_str} | Correct answer: {correct}
Predicted misconception: {pred_label}

Steps to fix:
1) Re-read what the question is asking.
2) Write down the correct mathematical rule.
3) Apply it step-by-step.
4) Check with a simpler example.

Self-check: Does your rule still work if the numbers change slightly?
"""

    fb = rule_based_verify(fb, ic, correct)
    fb = llm_verify(fb)
    return fb.strip()


# ============================================================
# DEMO: Generate feedback for 5 random samples using Qwen3-4B
# ============================================================

print("=" * 80)
print("  EXPLAINABLE AI — SAMPLE TEACHER FEEDBACK (Qwen3-4B)")
print("=" * 80)

# Add pred_top3 column if missing (use label_str as placeholder)
if 'pred_top3' not in train_df.columns:
    train_df['pred_top3'] = train_df['label_str']

rng = np.random.default_rng(42)
demo_idxs = rng.choice(len(train_df), size=min(5, len(train_df)), replace=False)

for idx in demo_idxs:
    row = train_df.iloc[idx]
    pred_top3_str = row.get("pred_top3", row.get("label_str", ""))
    top3 = str(pred_top3_str).split()[:3] if pd.notna(pred_top3_str) else []
    pred1 = top3[0] if top3 else "Unknown:Unknown"

    print(f"\n{'─'*80}")
    print(f"ROW {idx}")
    print(f"{'─'*80}")
    print(f"Question: {str(row['QuestionText'])[:150]}")
    print(f"Student Answer: {row['MC_Answer']}")
    print(f"Student Explanation: {str(row['StudentExplanation'])[:150]}")
    print(f"True Label: {row['label_str']}")
    print(f"Predictions: {top3}")

    correct_marker = "CORRECT" if (top3 and top3[0] == row["label_str"]) else "WRONG"
    print(f"Top-1 Match: {correct_marker}")

    print(f"\n--- Teacher Feedback (Qwen3-4B) ---")
    feedback = generate_teacher_feedback(row, pred1, top3)
    print(feedback)
    print()

# ============================================================
# CLEANUP Qwen3-4B EXPLAINER MODEL
# ============================================================

if explainer_model is not None:
    del explainer_model
    del explainer_tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("\nQwen3-4B explainer model cleaned up.")